# ImFusion CT Geometry

This notebook demonstrates CT geometry concepts using the ImFusion CT Python bindings: creating cone-beam data, configuring parametric geometry, editing detector properties, setting manual per-frame geometry, and converting to/from OpenGL/OpenCV matrices.


## Overview

In this tutorial you will learn how cone-beam CT geometry is represented in ImFusion and how to:
- Create ConeBeamMetadata for 2D X-ray frames
- Configure aquisition geometries. E.g. using parametric descriptions
- Describe detector properties (flat, fan, curved)
- Manually define per-frame geometry using transformation matrices
- Convert geometry to/from OpenGL and OpenCV camera matrices

#### Prerequisites
- A working `imfusion-sdk` and `imfusion-sdk-computed_tomography` installation with a valid license

#### Data
This tutorial uses small demo images (`regdemo0.png`, `regdemo1.png`) packaged with the repository.

## Setup

### Setup python path
This step can be skipped if the package got installed from PyPI.

In [1]:
import sys
import os
import numpy as np

# Add the build directory to Python path (adjust if needed)
build_lib_path = '/Users/wieczorek/Desktop/Dev/imfusionsuite/cmake-build-release/lib'
if build_lib_path not in sys.path:
    sys.path.insert(0, build_lib_path)


### Import the modules
We need to import `imfusion` and `imfusion.computed_tomography`.

In [2]:
try:
    import imfusion
    import imfusion.computed_tomography as ct
except ImportError as e:
    raise ImportError("Failed to import ImFusion CT bindings. Make sure the CT plugin is built and the path is correct.") from e

## Steps in this notebook
1) Load two demo frames and turn them into a `SharedImageSet` with cone-beam metadata
2) Enable the modern geometry and configure it parametrically
3) Explore detector properties (flat panel, fan-beam, cylindrical curvature)
4) Manually set per-frame geometry using explicit transforms
5) Convert geometry to/from OpenGL and OpenCV camera matrices
6) Compare with legacy `ConeBeamGeometry` (kept for backward compatibility)


### Load two demo frames and turn them into a `SharedImageSet` with cone-beam metadata

In [3]:
image_files = ["data/regdemo0.png", "data/regdemo1.png"]
shared_image_set = imfusion.SharedImageSet()
for path in image_files:
    img = imfusion.load(path)[0][0]
    img.spacing = [300.0 / img.descriptor.width, 300.0 / img.descriptor.height, 1.0]
    shared_image_set.add(img)

print(f"Loaded shared image set {shared_image_set}")
print(f"Is cone beam data: {ct.is_cone_beam_data(shared_image_set)}")
ct.make_cone_beam_data(shared_image_set)
print(f"Is cone beam data: {ct.is_cone_beam_data(shared_image_set)}")

Loaded shared image set imfusion.SharedImageSet(size: 2, [
    imfusion.SharedImage(USHORT width: 512 height: 512 spacing: 0.585938x0.585938x1 pix),
    imfusion.SharedImage(USHORT width: 512 height: 512 spacing: 0.585938x0.585938x1 pix)
])
Is cone beam data: False
Is cone beam data: True


### Enable the modern geometry and configure it parametrically

In [4]:
cone_beam_metadata = ct.ConeBeamMetadata.get(shared_image_set)
cone_beam_metadata.enable_modern_geometry()
param_gen = ct.ParametricGeometryGenerator()
param_gen.source_det_distance = 1200.0
param_gen.source_pat_distance = 700.0
param_gen.angle_range = 90.0
transform_setup = param_gen.transformation_setup
transform_setup.use_iso_center_parameters = True
transform_setup.iso_rotation = [-30.0, 0.0, 45.0]
cone_beam_metadata.add_generator(param_gen, select=True)

imfusion.show(shared_image_set)

### Explore detector properties (flat panel, fan-beam, cylindrical curvature)

In [5]:
detector_props = ct.DetectorPropertiesDataComponent.get_or_create(shared_image_set)
detector_props.curvature = ct.DetectorCurvature.FANFLAT
imfusion.show(shared_image_set)

detector_props.curvature = ct.DetectorCurvature.CYLINDRICAL
detector_props.curved_offsets = [np.array([0.4, 0.4]), np.array([0.4, 0.4])]
detector_props.radii = [800.0, 800.0]
imfusion.show(shared_image_set)

detector_props.curvature = ct.DetectorCurvature.FLAT

### Manually set per-frame geometry using explicit transforms

In [6]:
from scipy.spatial.transform import Rotation as R
for i in range(shared_image_set.size):
    source_data = ct.SourceDataComponent.get_or_create(shared_image_set, i)
    detector_data = ct.DetectorDataComponent.get_or_create(shared_image_set, i)
    source_data.location_source_in_detector_space = [0.0, 0.0, -1200.0]
    transform_matrix = np.eye(4)
    transform_matrix[0:3, 0:3] = R.from_euler('y', 90 * i / (shared_image_set.size - 1), degrees=True).as_matrix()
    detector_data.matrix_world_to_detector = transform_matrix

imfusion.show(shared_image_set)

### Convert geometry to/from OpenGL and OpenCV camera matrices

In [7]:
per_frame_geometries = ct.per_frame_geometry(shared_image_set)
for i, geom in enumerate(per_frame_geometries):
    print(f"\n--- Frame {i} ---")

    source_data_component = ct.SourceDataComponent.get(shared_image_set, i)
    print(f"Original source position: {source_data_component.location_source_in_detector_space}")
    detector_data_component = ct.DetectorDataComponent.get(shared_image_set, i)
    print(f"Original detector transformation: {detector_data_component.matrix_world_to_detector}")

    # Query OpenGL projection and modelview matrices
    pm =  geom.to_matrix_gl_image()
    print("OpenGL Projection Matrix:\n", pm)

    # Query OpenCV camera matrix components in pixel coordinates
    k, r, t =  geom.to_matrix_components_opencv_pixel()
    print("OpenCV Intrinsic Matrix (K):\n", k)
    print("OpenCV Rotation Matrix (R):\n", r)
    print("OpenCV Translation Vector (t):\n", t)

    # Query OpenCV projection matrix in pixel coordinates
    p = geom.to_matrix_opencv_pixel()
    print("OpenCV Projection Matrix (P):\n", p)

    # Try to reconstruct the geometry from OpenGL matrix
    # For this, we need the detector size (in mm) or pixel info
    # Let's use the detector size from the geometry if available
    # Here, we assume a square detector of 300x300 mm and 512x512 pixels for demonstration
    detector_size = shared_image_set[i].extent[0:2]
    width, height = shared_image_set[i].descriptor.dimensions[0:2]
    pixel_spacing = np.array([detector_size[0]/width, detector_size[1]/height])
    print(f"Detector size: {detector_size}, width: {width}, height: {height}, pixel_spacing: {pixel_spacing}")

    # Reconstruct geometry from OpenGL matrix (using detector size)
    geom_from_gl = ct.FullGeometryRepresentation.from_opengl_matrix(pm, detector_size)
    print("Reconstructed FullGeometryRepresentation from OpenGL matrix:\n", geom_from_gl)
    print(f"Reconstructed source position (from OpenGL matrix):\n{geom_from_gl.location_source_in_detector_space}")
    print(f"Reconstructed detector transformation (from OpenGL matrix):\n{geom_from_gl.matrix_world_to_detector}")

    # Reconstruct geometry from OpenGL matrix (using pixel info)
    geom_from_gl_pix = ct.FullGeometryRepresentation.from_opengl_matrix(pm, width, height, pixel_spacing)
    print("Reconstructed FullGeometryRepresentation from OpenGL matrix (pixel info):\n", geom_from_gl_pix)
    print(f"Reconstructed source position (from OpenGL matrix):\n{geom_from_gl_pix.location_source_in_detector_space}")
    print(f"Reconstructed detector transformation (from OpenGL matrix):\n{geom_from_gl_pix.matrix_world_to_detector}")

    # Reconstruct geometry from OpenCV projection matrix
    geom_from_cv = ct.FullGeometryRepresentation.from_opencv_matrix(p, width, height, pixel_spacing)
    print("Reconstructed FullGeometryRepresentation from OpenCV matrix:\n", geom_from_cv)
    print(f"Reconstructed source position (from OpenCV matrix):\n{geom_from_cv.location_source_in_detector_space}")
    print(f"Reconstructed detector transformation (from OpenCV matrix):\n{geom_from_cv.matrix_world_to_detector}")



--- Frame 0 ---
Original source position: [    0.     0. -1200.]
Original detector transformation: [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
OpenGL Projection Matrix:
 [[ 8.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -8.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.66666667e+00  1.20000000e+03]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00  1.20000000e+03]]
OpenCV Intrinsic Matrix (K):
 [[2.048e+03 0.000e+00 2.555e+02]
 [0.000e+00 2.048e+03 2.555e+02]
 [0.000e+00 0.000e+00 1.000e+00]]
OpenCV Rotation Matrix (R):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
OpenCV Translation Vector (t):
 [   0.    0. 1200.]
OpenCV Projection Matrix (P):
 [[2.048e+03 0.000e+00 2.555e+02 3.066e+05]
 [0.000e+00 2.048e+03 2.555e+02 3.066e+05]
 [0.000e+00 0.000e+00 1.000e+00 1.200e+03]]
Detector size: [300. 300.], width: 512, height: 512, pixel_spacing: [0.5859375 0.5859375]
Reconstructed FullGeometryRepresentation fro

### Compare with legacy `ConeBeamGeometry`

Note: We are transitioning fully to the modern geometry representation but currently there are some algorithms remaining, that require the legacy `ConeBeamGeometry`.

In [8]:
# Disable modern geometry and use legacy geometry
cone_beam_metadata.disable_modern_geometry()
legacy_geom = cone_beam_metadata.geometry()
legacy_geom.source_det_distance = 1000.0
legacy_geom.source_pat_distance = 500.0
legacy_geom.det_size_x = 300.0
legacy_geom.det_size_y = 300.0
legacy_geom.angle_range = 90.0
legacy_geom.recon_rot_x = 0.0
legacy_geom.recon_rot_y = 0.0
legacy_geom.angle_start = 45.0
legacy_geom.recon_offset_x = 0.0
legacy_geom.recon_offset_y = 0.0
legacy_geom.recon_offset_z = 0.0

imfusion.show(shared_image_set)